In [2]:
import pandas as pd
import datasets
import unicodedata
import re


In [3]:
sporsett_dataset = datasets.load_dataset(
    "parquet", 
    data_files={
        "train": "https://huggingface.co/datasets/GEM/sportsett_basketball/resolve/refs/convert/parquet/default/train/*.parquet", 
        "validation": "https://huggingface.co/datasets/GEM/sportsett_basketball/resolve/refs/convert/parquet/default/validation/*.parquet", 
        "test": "https://huggingface.co/datasets/GEM/sportsett_basketball/resolve/refs/convert/parquet/default/test/*.parquet"
    }
)

In [68]:
def create_game_summaries_dataset(dataset):
    game_summaries = []
    print(dataset)

    for row in dataset.select(range(2)):
        game_id = row.get("sportsett_id")

        if "target" in row and row["target"]:
            summary = str(row["target"])
        elif "summaries" in row and isinstance(row["summaries"], list) and len(row["summaries"] > 0):
            summary = str(row["summaries"][0])

        sportsett_id_lookup = {row['sportsett_id']: row for row in dataset}

        # Home 
        home_name = row['teams']['home']['name']
        home_points = int(row['teams']['home']['line_score']['game']['PTS'])
        home_next_game_id = row['teams']['home']['next_game_id']
        if home_next_game_id in sportsett_id_lookup:
            home_next_game_row = sportsett_id_lookup[home_next_game_id]
            home_next_opponent = home_next_game_row['teams']['home']['name'] if home_next_game_row['teams']['home']['name'] != home_name else home_next_game_row['teams']['vis']['name']


        # Visitor
        vis_name = row['teams']['vis']['name']
        vis_points = int(row['teams']['vis']['line_score']['game']['PTS'])
        vis_next_game_id = row['teams']['vis']['next_game_id']
        if vis_next_game_id in sportsett_id_lookup:
            vis_next_game_row = sportsett_id_lookup[vis_next_game_id]
            vis_next_opponent = vis_next_game_row['teams']['home']['name'] if vis_next_game_row['teams']['home']['name'] != vis_name else vis_next_game_row['teams']['vis']['name']


        (winner_team, winner_points, loser_team, loser_points, win_home_or_vis, loss_home_or_vis, winner_next_opponent) = (
            (home_name, home_points, vis_name, vis_points, 'home', 'visitor', home_next_opponent)
            if home_points > vis_points
            else (vis_name, vis_points, home_name, home_points, 'visitor', 'home', vis_next_opponent)
        )

        total_points = home_points + vis_points
        margin = abs(home_points - vis_points)

        game_summaries.append({
            'sporsett_id': game_id,
            "summary": summary,
            "winner_team": winner_team,
            "winner_points": winner_points,
            "loser_team": loser_team,
            "loser_points": loser_points,
            "win_home_or_vis": win_home_or_vis,
            "loss_home_or_vis": loss_home_or_vis,
            "winner_next_opponent": winner_next_opponent,   
            "def_or_off": "defensive" if total_points < 210 else "offensive" ,
            "double_digit_margin": (10 <= margin < 100),
            "total_points_gt_180": total_points > 180
        })

    return game_summaries


In [ ]:
summaries = create_game_summaries_dataset(sporsett_dataset['test'])
summaries

Dataset({
    features: ['sportsett_id', 'gem_id', 'game', 'teams', 'summaries', 'target', 'references', 'linearized_input'],
    num_rows: 1230
})
